In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, date_format, substring, when, lit
from datetime import datetime, date
from dateutil.relativedelta import relativedelta

spark.sql("USE CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA silver")

In [0]:
df_bronze = spark.table("bronze.uk_hpi")


print(f"Rows read from bronze: {df_bronze.count()}")

In [0]:
df_scotland = df_bronze.filter(col("AreaCode").startswith("S"))

print(f"Rows after Scotland filter: {df_scotland.count()}")

In [0]:
df_dated = df_scotland \
    .withColumn("Date", to_date(col("Date"), "dd/MM/yyyy"))\
    .withColumn("year_month", date_format(col("Date"), "yyyy-MM"))

df_dated.select("RegionName","AreaCode", "Date", "year_month", "AveragePrice").show(5)

In [0]:
df_clean = df_dated.withColumn(
    "SalesVolume",
    when(col("SalesVolume").isNull(), lit(0)).otherwise(col("SalesVolume"))
)

df_clean.filter(col("AreaCode") == "S92000003").select("Date", "SalesVolume").show(5)

In [0]:
df_silver = df_clean.withColumn("SalesVolume", col("SalesVolume").cast("long"))

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.uk_hpi")
)

print(f"Written {df_silver.count()} rows to silver.uk_hpi")

In [0]:
spark.sql("""
    SELECT RegionName, year_month, AveragePrice, SalesVolume
    FROM silver.uk_hpi
    WHERE AreaCode = 'S92000003'
    ORDER BY year_month DESC
    LIMIT 5
""").show()

In [0]:
# Silver transformation complete
# Source: bronze.uk_hpi
# Rows in: 149,895 | Rows after Scotland filter: 9,207 | Rows written: 9,207
# Transformations applied:
#   - Filtered to AreaCode LIKE 'S%' (Scotland national + 32 local authorities)
#   - Cast Date from string dd/MM/yyyy to date type
#   - Derived year_month join key (yyyy-MM string)
#   - Null SalesVolume replaced with 0, cast to long
# Table: silver.uk_hpi

print("silver_01_uk_hpi complete.")